# About this template {.unnumbered}

This is the **web version** of the JTH lab report template. It gives you a
single HTML file that you can email or hand in, and that can hold what paper
cannot: animations, film, sound and graphs you can zoom into.

::: {.callout-tip}
## Which version should I pick?
| | `report.ipynb` (PDF) | `report-html.ipynb` (this one) |
|---|---|---|
| Handed in on paper, printed | yes | no |
| Static figures, equations, references | yes | yes |
| Animations, film, sound | no | yes |
| Interactive graphs (zoom, hover) | no | yes |
| Code the reader can unfold | no | yes |

Always follow the instructions for your course. If nothing is said: hand in a PDF.
:::

Render it with the **Render to html using Quarto** button in the toolbar, or
**Render & preview** to see the result straight away. Thanks to
`embed-resources: true` everything ends up in a single file, with no images or
films that can be left behind.

Delete this section when you write your own report.

# Introduction

Describe the assignment, the purpose, and what the reader should get out of the
report. Cite your sources with square brackets, as in `[@Stromberg2012]` gives
[@Stromberg2012].

The citation style is IEEE (numbered, [1]). For APA (author and year), switch the
`csl:` line in the YAML block in the first cell; the comment there shows how.

Maths is written as in the PDF version. Inside a sentence: $F = ma$. On its own
line, with a label that can be referenced:

$$
\int_0^1 x^2\,dx = \frac{1}{3}
$$ {#eq-integral}

See @eq-integral.

# Theory

Short, but enough for the reader to follow the calculations. A callout box is
often useful:

::: {.callout-note}
## Assumption
Air resistance is neglected throughout the report.
:::

# Method and results

## Figures from code

Just as in the PDF version: give the cell a label and a caption, and the figure
is numbered and can be referenced with `@fig-sine`.

In [ ]:
#| label: fig-sine
#| fig-cap: "A sine wave generated with Python"
#| code-fold: show
import numpy as np
import matplotlib.pyplot as plt

t = np.linspace(0, 4 * np.pi, 400)
plt.figure(figsize=(6, 3))
plt.plot(t, np.sin(t))
plt.xlabel("t [s]")
plt.ylabel("u(t) [V]")
plt.grid(alpha=.3)
plt.show()

See @fig-sine. With `code-fold: show` the code is visible with an arrow for
folding it away; `#| code-fold: true` starts folded and `#| echo: false` hides
it completely.

## Tables

A pandas table becomes an HTML table. `#| tbl-cap` gives it a caption and a
number.

In [ ]:
#| label: tbl-measurements
#| tbl-cap: "Measured values for three trials"
import pandas as pd

df = pd.DataFrame({
    "Trial": [1, 2, 3],
    "Mass m [kg]": [0.50, 0.75, 1.00],
    "Force F [N]": [4.91, 7.36, 9.81],
})
df["a = F/m [m/s²]"] = (df["Force F [N]"] / df["Mass m [kg]"]).round(2)
df

See @tbl-measurements.

# Moving pictures

Three ways to get an animation into the report. All three build on
`matplotlib.animation.FuncAnimation`; the difference is how it is saved. The
examples follow the course material at
[python.ju.se](https://python.ju.se/ProgrammingFundamentals/RichMediaReports.html).

## Way 1: GIF, the simplest

Works everywhere, but the file gets large and the colours are limited (256 of
them). Good for short, simple sequences.

In [ ]:
#| label: gif-animation
#| echo: true
#| output: false
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation

fig, ax = plt.subplots(figsize=(4, 4))
line, = ax.plot([], [], "ro")
ax.set_xlim(-1.5, 1.5)
ax.set_ylim(-1.5, 1.5)
t = np.linspace(0, 2 * np.pi, 60)
ax.plot(np.cos(t), np.sin(t), "b-")
ax.set_aspect("equal", "box")


def update(frame):
    line.set_data([np.cos(frame)], [np.sin(frame)])
    return line,


ani = animation.FuncAnimation(fig, update, frames=t, blit=True)
plt.close()
ani.save("circle_animation.gif", writer="pillow", fps=20, dpi=80)

![A point travelling around the unit circle (GIF)](circle_animation.gif){width=60%}

## Way 2: MP4, the best quality

`ffmpeg` is installed on the server. The film is small and sharp, and the
reader gets controls for play, pause and seek.

In [ ]:
#| label: mp4-animation
#| echo: true
#| output: false
fig, ax = plt.subplots(figsize=(4, 4))
line, = ax.plot([], [], "ro")
ax.set_xlim(-1.5, 1.5)
ax.set_ylim(-1.5, 1.5)
t = np.linspace(0, 2 * np.pi, 240)
ax.plot(np.cos(t), np.sin(t), "b-")
ax.set_aspect("equal", "box")
ani = animation.FuncAnimation(fig, update, frames=t, blit=True)
plt.close()
ani.save("circle_animation.mp4", writer="ffmpeg", fps=30, dpi=150)

<video width="60%" controls autoplay loop muted>
  <source src="circle_animation.mp4" type="video/mp4">
  Your browser cannot show the film.
</video>

## Way 3: interactive playback

`to_jshtml` builds a player with a frame selector, so the reader can step back
and forth. Note `#| output: asis`: the HTML has to go in as it is.

In [ ]:
#| label: jshtml-animation
#| output: asis
#| code-fold: true
from IPython.display import HTML

fig, ax = plt.subplots(figsize=(3.2, 3.2))
line, = ax.plot([], [], "ro")
ax.set_xlim(-1.5, 1.5)
ax.set_ylim(-1.5, 1.5)
t = np.linspace(0, 2 * np.pi, 30)
ax.plot(np.cos(t), np.sin(t), "b-")
ax.set_aspect("equal", "box")
ani = animation.FuncAnimation(fig, update, frames=t, blit=True)
plt.close()

HTML(f'''<div style="width:100%">
<style>img {{ max-width: 100% !important; }}</style>
{ani.to_jshtml(default_mode="loop")}
</div>''')

::: {.callout-warning}
## Size
An animation can weigh a few MB, and with `embed-resources: true` all of it
ends up in the HTML file. Keep the animations short (a couple of seconds) and
the `dpi` moderate, or the file gets too heavy to email.
:::

# Interactive graphs

With `plotly` the reader can zoom, pan and read off values by pointing at the
graph. Useful when the measurement series is dense or when details disappear in
a static picture.

In [ ]:
#| label: fig-plotly
#| fig-cap: "Damped oscillation. Point at the graph to read off values"
#| code-fold: true
import numpy as np
import plotly.graph_objects as go
import plotly.io as pio

# "notebook" embeds the drawing library in the file, so the graph works
# without internet. Switch to "notebook_connected" if you want a small file.
pio.renderers.default = "notebook"

t = np.linspace(0, 10, 500)
x = np.exp(-0.3 * t) * np.cos(2 * np.pi * t)

fig = go.Figure()
fig.add_trace(go.Scatter(x=t, y=x, mode="lines", name="x(t)"))
fig.add_trace(go.Scatter(x=t, y=np.exp(-0.3 * t), mode="lines",
                         name="envelope", line=dict(dash="dash")))
fig.update_layout(template="simple_white", height=380,
                  xaxis_title="t [s]", yaxis_title="x(t) [m]",
                  margin=dict(l=60, r=20, t=30, b=50))
fig

See @fig-plotly.

::: {.callout-note collapse="true"}
## Plotly makes the file about 5 MB bigger
The drawing library is embedded (`pio.renderers.default = "notebook"`) so that
the graph works without internet. If you would rather have a small file, and can
count on the reader being online, change that line to:

```python
pio.renderers.default = "notebook_connected"
```

If you do not need the interactivity, matplotlib is enough and the file is only
a few hundred kB.
:::

# Sound

A sound says more than a spectrum plot when it comes to beats or vibrations,
for instance.

In [ ]:
#| label: sound
#| code-fold: true
import numpy as np
from IPython.display import Audio

fs = 22050
t = np.linspace(0, 2, 2 * fs, endpoint=False)
# two nearby frequencies produce beats
signal = 0.4 * (np.sin(2 * np.pi * 220 * t) + np.sin(2 * np.pi * 223 * t))
Audio(signal, rate=fs)

# Tabs and other things that only exist on the web

::: {.panel-tabset}
## Result
A short summary for the reader who only wants the answer.

## Calculation
$$ a = \frac{F}{m} = \frac{9.81}{1.00} = 9.81\ \mathrm{m/s^2} $$

## Code
```python
a = F / m
```
:::

::: {.callout-note collapse="true"}
## Flow charts (add about 3 MB)
Quarto can draw diagrams directly in the text with ```` ```{mermaid} ````. It
looks good, but the drawing library is embedded and the file grows by roughly
3 MB, so the template does not use it by default.
:::

# Discussion and conclusion

Interpret the results, compare them with the theory, and be clear about the
sources of error.

# How to share the report

Rendering gives you **one** file, `report-html.html`, with everything baked in:

* download it (right-click the file in the file browser, then Download) and email it, or
* hand it in where the course says.

The template as it stands gives roughly 11 MB, half of which is plotly. If the
file is too heavy to email:

* delete the sections you do not use (plotly weighs about 5 MB, the animations about 1 MB),
* shorten the animations and lower the `dpi`,
* or set `embed-resources: false`, which puts films and images in the folder
  `report-html_files/`, which then has to travel with the file.